# Disaster relief thesis review: reproduce selected claims

Run from a trusted notebook copy. Put the unmodified evidence directory beside this notebook
as `evidence/`, start the Jupyter kernel with this directory as its working directory, or set
`THESIS_REVIEW_EVIDENCE` to an absolute folder path before starting Jupyter. The evidence source
is an exact manifest-pinned selection from GitHub, and is verified in full before metric parsing.
No network, additional Python packages, model training, full baseline run, or FY2025 outcomes are
needed. Historical FEMA obligations are administrative records, not economic damage or humanitarian need.

**Scope.** Where possible this notebook recomputes arithmetic from aggregated observations or matched
case summaries. Reported statistics that cannot be recomputed from those inputs are labelled archived.
It does not certify model independence, show operational skill, or evaluate a human participant.


## Authenticate the offline evidence


In [ ]:
authenticated = False
claims = {}
evidence = {}

import csv
import hashlib
import io
import json
import os
from pathlib import Path
import re
from statistics import mean

# Pin over the manifest bytes, not a trust-on-first-use list generated from the input.
MANIFEST_SHA256 = "006773bfcb065f426fd79384c7d275fe102eda2c0b1450d286a4a82a9510cb32"
evidence_dir = Path(os.environ.get("THESIS_REVIEW_EVIDENCE", str(Path.cwd() / "evidence"))).resolve(strict=True)

def require(ok, message):
    if not ok:
        raise ValueError(message)

manifest_path = evidence_dir / "manifest.json"
require(not manifest_path.is_symlink(), "Manifest must be a regular file")
manifest_bytes = manifest_path.read_bytes()
require(hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256,
        "Manifest SHA-256 mismatch; no evidence was used")
manifest = json.loads(manifest_bytes)
require(manifest["schema_version"] == 1 and manifest["repository"] == "leonmarienga/ML-thesis",
        "Unexpected evidence schema or repository")
verified = {}
for member in manifest["files"]:
    name = member["name"]
    require(bool(re.fullmatch(r"[a-z][a-z_0-9]*\.(csv|json|md)", name)) and name not in verified,
            "Unsafe or duplicate evidence name")
    require(bool(re.fullmatch(r"[0-9a-f]{40}", member["commit"])) and
            member["path"].startswith("research/") and ".." not in member["path"].split("/"),
            f"Unpinned or unsafe source path for {name}")
    source = evidence_dir / name
    require(source.is_file() and not source.is_symlink(), f"Evidence unavailable or not a regular file: {name}")
    data = source.read_bytes()
    blob = hashlib.sha1(f"blob {len(data)}\0".encode("ascii") + data).hexdigest()
    require(member["size"] == len(data) and member["sha256"] == hashlib.sha256(data).hexdigest() and
            member["git_blob"] == blob, f"Evidence digest mismatch: {name}; no metrics calculated")
    verified[name] = data

def rows(name):
    return list(csv.DictReader(io.StringIO(evidence[name].decode("utf-8-sig"), newline="")))

def one(items, **where):
    matched = [r for r in items if all(r[k] == v for k, v in where.items())]
    require(len(matched) == 1, f"Expected an unambiguous row for {where}")
    return matched[0]

def near(a, b, tolerance=1e-8):
    return abs(a-b) <= tolerance

evidence = verified
authenticated = True
print(f"Verified {len(evidence)} pinned files against SHA-256, byte counts and Git blob IDs before parsing")


## Track A historical archive (not a forecast)


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
historic = rows("track_a_bands.csv")
bands = ["0-100K", "100K-1M", "1M-50M", "50-200M", "200-500M", "500M+"]
require([r["actual_band"] for r in historic] == bands, "Track A bands changed")
totals = [sum(int(r[b]) for b in bands) for r in historic]
hits = [int(r[b]) for r, b in zip(historic, bands)]
require(totals == [int(r["actual_n"]) for r in historic] and (sum(hits), sum(totals)) == (830, 912),
        "Track A confusion counts mismatch")
require(all(near(100*k/n, float(r["recall_pct"]), 1e-7) for k, n, r in zip(hits, totals, historic)),
        "Track A recalls mismatch")
versions = {r["amount_version"]: float(r["mae_usd"]) for r in rows("track_a_amounts.csv")}
require(len(versions) == 3 and near(versions["challenger_stack_prediction"], 5014819.011614494, 0.01),
        "Track A amount version mismatch")
claims["track_a"] = {"scope":"archived reconstruction; target-conditioned rules and non-forward LFYO",
                     "correct":sum(hits), "total":sum(totals), "accuracy_pct":round(100*sum(hits)/sum(totals), 2),
                     "band_totals":totals, "band_hits":hits, "archived_amount_mae_usd":versions}
print(f"Track A archive: {sum(hits)}/{sum(totals)} = {100*sum(hits)/sum(totals):.2f}%; not forward skill.")
for key, usd in versions.items():
    print(f"Reported {key}: MAE ${usd:,.2f}; read from the pinned aggregate, not re-scored.")


## Matched forward comparison on examined historical cases


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
fwd = rows("forward_t5.csv")
require([r["actual_band"] for r in fwd] == bands, "Forward matrix bands changed")
n = [int(r["actual_n"]) for r in fwd]
require(sum(n) == 376 and min(n) == 1, "Unexpected matched case support")
forward = {}
for label in ("retrospective_metadata", "verified_link"):
    d = [int(r[label+"_hits"]) for r in fwd]
    recall = [k/t for k, t in zip(d, n)]
    require(all(near(p, float(r[label+"_recall"])) for p, r in zip(recall, fwd)),
            "Archived per-class forward recall mismatch")
    forward[label] = {"correct":sum(d), "accuracy_pct":round(100*sum(d)/sum(n), 2),
                      "balanced_accuracy_pct":round(100*mean(recall), 2)}
require(forward["verified_link"]["accuracy_pct"] > forward["retrospective_metadata"]["accuracy_pct"] and
        forward["verified_link"]["balanced_accuracy_pct"] < forward["retrospective_metadata"]["balanced_accuracy_pct"],
        "Forward trade-off changed")
claims["matched_forward"] = {"scope":"same retrospective held cases, not an independent prospective release",
                             "n":sum(n),"band_support":n, "metrics": forward}
print("Same 376 matched cases; raw and class-balanced accuracy (%):")
for label, scores in forward.items():
    print(f"{label}: {scores['accuracy_pct']:.2f}, {scores['balanced_accuracy_pct']:.2f}")
matrix = rows("forward_matrices.csv")
variants = {"retrospective_metadata":"ablation_retrospective_metadata", "verified_link":"ablation_verified_link_only"}
for label, variant in variants.items():
    counts = {(r["actual_band"], r["predicted_band"]): int(r["n"]) for r in matrix if r["variant"] == variant}
    require(set(counts) == {(x,y) for x in bands for y in bands}, "Incomplete forward confusion matrix")
    require([sum(counts[b,p] for p in bands) for b in bands] == n and
            [counts[b,b] for b in bands] == [int(r[label+"_hits"]) for r in fwd],
            "T5 and source forward matrices disagree")
compare = one(rows("forward_bootstrap.csv"), variant="ablation_verified_link_only")
mae = float(compare["mae_difference_usd"])
mae_ci = [float(compare[k]) for k in ("mae_ci_low", "mae_ci_high")]
require(mae_ci[0] < 0 < mae < mae_ci[1] and
        near(float(compare["accuracy_difference"]), (224-192)/376),
        "Archived bootstrap contrast changed")
claims["matched_forward"].update(archived_mae_change_usd=mae, archived_mae_ci95_usd=mae_ci)
print(f"Archived episode-bootstrap MAE change ${mae:,.0f}: its interval crosses zero.")


## Uncertainty coverage and upper-tail failure


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
intervals = rows("interval_t8.csv")
cts = ["n","covered","above_upper","below_lower"]
interval_report = {}
require(len(intervals) == 12, "Unexpected interval rows")
for level in ("0.80000000", "0.90000000"):
    for pipeline in ("residual", "cqr"):
        matched = [one(intervals, nominal=level, pipeline=pipeline, scope=s)
                   for s in ("all", "under_50m", "50m_or_more")]
        counts = [{k:int(r[k]) for k in cts} for r in matched]
        require(all(c["n"] == c["covered"]+c["above_upper"]+c["below_lower"] for c in counts),
                "Interval hit/miss counts do not reconcile")
        require(all(counts[0][k] == counts[1][k]+counts[2][k] for k in cts) and
                [c["n"] for c in counts] == [308,299,9], "Interval strata do not aggregate")
        label = f"{float(level):.0%} {pipeline}"
        interval_report[label] = {r["scope"]:{**c,"coverage_pct":round(100*c["covered"]/c["n"],2),
                                             "median_width_usd":float(r["median_width_usd"])}
                                  for r,c in zip(matched,counts)}
        print(f"{label}: overall {counts[0]['covered']}/308, >=$50M {counts[2]['covered']}/9")
require(interval_report["90% residual"]["all"]["covered"] == 281 and
        interval_report["90% residual"]["50m_or_more"]["covered"] == 0, "90% residual finding changed")
claims["intervals"] = {"scope":"archived matched aggregate; very small tail, no out-of-sample guarantee",
                       "results":interval_report}
print("A nominal level is not a tail guarantee; CQR tail breadth and support matter.")
audit = rows("uncertainty_review.csv")
for scope, key in (("all","all"), ("below_50m","under_50m"), ("at_least_50m","50m_or_more")):
    original = interval_report["90% residual"][key]
    independent_aggregate = one(audit, scope=scope)
    require(all(int(independent_aggregate[k]) == original[k] for k in ("n","covered")),
            "90% interval aggregate did not reconcile to audit table")
require(int(one(audit,scope="all")["flagged_misses"]) == 23 and
        int(one(audit,scope="at_least_50m")["misses"]) == 9, "Retrospective flag counts changed")


## Transportation and governance trade-offs (synthetic only)


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
metrics = ("overall","high_vulnerability","least_served")
archived_floors = rows("transport_floors.csv")
floor = {}
for metric in metrics:
    f = one(archived_floors, policy="vulnerability", metric=metric)
    delta = float(f["difference"])
    lo, hi = float(f["lower_95"]), float(f["upper_95"])
    require(lo <= delta <= hi, "Invalid archived floor range")
    floor[metric] = {"change_pp":100*delta, "archived_interval_pp":[100*lo,100*hi]}
require(floor["least_served"]["change_pp"] > 0 > floor["high_vulnerability"]["change_pp"],
        "Floor policy trade-off changed")
print("Batch 14 synthetic geographic floor (0 to 100% of reference), differences in percentage points:")
for k, v in floor.items():
    print(f"  {k}: {v['change_pp']:+.2f}")
claims["transport_floor"] = {"scope":"archived simulator contrasts, not observed service", "metrics":floor}
pairs = rows("veto_contrasts.csv")
arm = "optimizer_with_scripted_veto"
require(len(pairs) == 9, "Unexpected veto aggregate shape")
for r in pairs:
    require(sum(int(r[k]) for k in ("positive","tied","negative")) == int(r["n"]) == 48,
            "Veto contrast matched support invalid")
    require(near(float(r["left_mean"])-float(r["right_mean"]), float(r["difference"])),
            "Veto contrast means do not reconcile")
veto = {}
for metric in metrics:
    r = one(pairs, left=arm, right="optimizer_only", metric=metric)
    delta = float(r["left_mean"])-float(r["right_mean"])
    lo, hi = float(r["lower_95"]), float(r["upper_95"])
    require(lo <= delta <= hi, "Invalid archived veto range")
    veto[metric] = {"change_pp":100*delta, "archived_interval_pp":[100*lo,100*hi]}
require(veto["least_served"]["change_pp"] > 0 > veto["overall"]["change_pp"] and
        veto["high_vulnerability"]["change_pp"] < 0, "Veto trade-off changed")
print("Separate 48-world synthetic optimizer+scripted veto versus optimizer, percentage points:")
for k, v in veto.items():
    print(f"  {k}: {v['change_pp']:+.2f}")
claims["policy_veto"] = {"scope":"synthetic scripted governance, not participants or operational effect", "n":48, "metrics":veto}


## Independent-validation limits


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
from datetime import datetime, timedelta

folds = rows("target_blind_folds.csv")
require([int(r["origin_fy"]) for r in folds] == list(range(2019, 2025)), "Unexpected origins")
require(sum(int(r["n_held"]) for r in folds) == 376, "Target-blind held support changed")
for r in folds:
    require(int(r["max_train_fy"]) < int(r["origin_fy"]) and
            r["max_train_end_or_start"] < r["min_held_start_or_declaration"] and
            int(r["held_groups_in_training"]) == int(r["train_held_declaration_id_intersection"]) == 0,
            "Archived fold chronology/group check failed")
    require(r["permutation_invariance"] == r["reassignment_invariance"] == "True" and
            min(int(r["permutation_held_labels_changed"]), int(r["reassignment_held_labels_changed"])) > 0,
            "Archived held-outcome stress check failed")
tb = json.loads(evidence["target_blind_summary.json"])
require(tb["status"] == "exploratory_development_only" and tb["fy_2025_accessed"] is False and
        tb["held_outcome_stress_origins_passed"] == len(folds) and
        tb["independent_replication_priority_gate_met"] is False, "Target-blind gate changed")
gain = tb["candidate"]["correct"]-tb["baseline"]["correct"]
changed = tb["all_coverage"]["changed_n"]
groups = tb["all_coverage"]["changed_group_n"]
require((gain,changed,groups) == (1,3,2), "Target-blind qualification changed")
claims["target_blind"] = {"scope":"archived noninterference check records; examined cohort, not replication",
                          "origins":len(folds), "net_correct":gain, "changed":changed,
                          "changed_groups":groups, "promotion_gate_met":False}
print("Target-blind stress records pass six earlier-vs-held folds; one net correction, two changed groups. Not promoted.")

weights = rows("weight_metrics.csv")
a,b = [one(weights, arm=arm, rank="0") for arm in ("real_only","weighted_20x")]
require(a["n"] == b["n"] == "135" and a["tail_n"] == b["tail_n"] == "11", "Unexpected signed-action support")
tails = rows("weight_bootstrap.csv")
r = one(tails, endpoint="signed_elapsed", arm="weighted_20x", metric="tail_mae")
delta = float(b["tail_mae"])-float(a["tail_mae"])
require(near(delta, float(r["change"]), 0.01) and float(r["lower_95"]) < 0 < float(r["upper_95"]),
        "20× tail comparison or confidence interval changed")
alerts = [int(x["alerts_false_10pct"]) for x in (a,b)]
require(alerts == [13,39], "20× false alert counts changed")
claims["weight_20x"] = {"scope":"exploratory matched elapsed signed-action cohort; not independent",
                        "n":135, "tail_n":11, "tail_mae_change_usd":delta,
                        "archived_ci95_usd":[float(r["lower_95"]),float(r["upper_95"])],
                        "false_alerts_baseline_then_weighted":alerts}
print(f"20× signed-action candidate: tail MAE difference ${delta:,.0f}; CI crosses zero; false alerts 13 to 39.")

gauge_archive = json.loads(evidence["gauge_stage_summary.json"])
gauge_pairs = rows("gauge_stage_pairs.csv")
crossing = datetime.fromisoformat(gauge_archive["first_actual_15m_crossing_12ft_utc"])
require(gauge_archive["episodes"] == 1 and len(gauge_pairs) == 230 and crossing.isoformat() == "2022-12-31T09:00:00+00:00",
        "Gauge cohort changed")
gauge = []
for a in gauge_archive["cases"]:
    issue = datetime.fromisoformat(a["bulletin_wmo_issue_utc"])
    pairs = [r for r in gauge_pairs if r["wmo_issue_utc"] == issue.isoformat()]
    valid = [datetime.fromisoformat(r["valid_utc"]) for r in pairs]
    require(len(valid) == len(set(valid)) == 115 and valid == sorted(valid) and
            all(t > issue and (t-issue).total_seconds() == int(r["lead_minutes"])*60 for t,r in zip(valid,pairs)) and
            all(t-s == timedelta(hours=1) for s,t in zip(valid,valid[1:])),
            "Gauge issue/time support failed")
    fc = [float(r["forecast_stage_ft"]) for r in pairs]
    obs = [float(r["approved_gage_height_ft"]) for r in pairs]
    first = next(t for t,h in zip(valid, fc) if h >= 12)
    err = mean(abs(x-y) for x,y in zip(fc,obs))
    require(near(err,a["mae_ft"]) and max(fc) == a["forecast_peak_ft"] and max(obs) == a["observed_peak_at_forecast_hours_ft"] and
            first.isoformat() == a["first_forecast_hour_at_or_above_12_ft_utc"], "Gauge summary did not reproduce")
    precedes = issue < crossing
    gauge.append({"issue_utc":issue.isoformat(), "issue_preceded_observed_crossing":precedes,
                  "forecast_delay_hours_if_pre_crossing":(first-crossing).total_seconds()/3600 if precedes else None,
                  "future_hours":len(valid), "mae_ft":err,"forecast_peak_ft":max(fc),"observed_peak_ft":max(obs)})
require(len(gauge) == 2 and gauge[0]["forecast_delay_hours_if_pre_crossing"] == 14 and
        gauge[0]["forecast_peak_ft"] == 13 and gauge[0]["observed_peak_ft"] == 16.6 and
        gauge[1]["issue_preceded_observed_crossing"] is False, "Gauge crossing scope changed")
claims["gauge_stage"] = {"scope":"two issues, one historical event; no independent skill or obligation inference",
                         "episodes":1,"archived_actual_crossing_utc":crossing.isoformat(),"issues":gauge}
print("Gauge record: 115 future hours per issue, one event. Early crossing forecast 14 h late (peak 13.0 vs 16.6 ft); late issue was after crossing.")


## Prospective protocol dry-run record


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
tabletop = json.loads(evidence["tabletop_dry.json"])
require(tabletop["participants"] == 0 and tabletop["status"] == "simulation_only", "Tabletop scope drift")
require((tabletop["schedule_slots"], tabletop["scenario_ids"],
         tabletop["balanced_exposures_per_scenario_condition"], tabletop["closed_zone_retained"]) ==
        (4,6,2,True), "Archived tabletop invariant mismatch")
claims["tabletop"] = {"scope": "archived schedule dry run, not research with people", **tabletop}
print("Tabletop record: four schedule slots, six scenario IDs, zero participants.")


## Machine-readable claim ledger


In [ ]:
require(authenticated, "Evidence is not verified; run the bootstrap in a fresh kernel")
print(json.dumps(claims, indent=2, sort_keys=True, allow_nan=False))
